# 📈 MMM Project: Model Fitting and Diagnostics

In this notebook, we fit our Bayesian Media Mix Model using PyMC-Marketing. We perform MCMC sampling to estimate adstock decay, saturation points, and channel coefficients while verifying model convergence.

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import arviz as az
from modeling import MMMWrapper
from utils.logging_config import setup_logging

logger = setup_logging("mmm_modeling")
logger.info("Starting MMM fitting pipeline")

# Load data
df = pd.read_csv('../data/simulated_data.csv')
media_cols = ['linear_tv_spend', 'ctv_spend', 'paid_search_spend', 'paid_social_spend', 'display_spend']
control_cols = ['sin_period_1', 'cos_period_2'] # plus holiday columns in practice
target_col = 'conversions'
date_col = 'week'

wrapper = MMMWrapper(df, target_col, date_col, media_cols, control_cols)
logger.info("Model initialized, beginning MCMC")

### 1. Bayesian Model Estimation

We sample with specific priors (HalfNormal for impact, Beta for adstock) to ensure physical constraints on the marketing inputs.

In [ ]:
# fit the model (using 1000 tuning/draws for faster demo execution)
idata = wrapper.fit(tune=1000, draws=1000, chains=2)
logger.info("MCMC sampling complete")

### 2. Convergence Diagnostics

We use trace plots and R-hat statistics to verify that the MCMC chains have mixed well and converged to the target posterior.

In [ ]:
import matplotlib.pyplot as plt
az.plot_trace(idata, var_names=["adstock_alpha", "saturation_lam", "saturation_beta"])
plt.show()

### 3. Posterior Summary Table

R-hat values for all critical parameters must be < 1.05.

In [ ]:
summary = az.summary(idata, var_names=["adstock_alpha", "saturation_lam", "saturation_beta"])
summary